# Online Sales Intelligence with SQL

## Project Overview

This project analyzes online marketplace transactions using SQL to identify:

- Overall sales performance
- Monthly revenue trends
- Product category performance
- Regional sales patterns
- Payment method preferences
- Top-performing products

## Tools

- SQL
- DuckDB
- Kaggle Notebook
- AI-assisted query review

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shreyanshverma27/online-sales-dataset-popular-marketplace-data")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/shreyanshverma27/online-sales-dataset-popular-marketplace-data


In [2]:
import glob
import duckdb

csv_files = glob.glob(
    "/kaggle/input/datasets/shreyanshverma27/online-sales-dataset-popular-marketplace-data/Online Sales Data.csv"
)

csv_path = csv_files[0]

con = duckdb.connect()

print(f"Dataset: {csv_path}")

Dataset: /kaggle/input/datasets/shreyanshverma27/online-sales-dataset-popular-marketplace-data/Online Sales Data.csv


In [3]:
con.execute(f"""
CREATE OR REPLACE VIEW online_sales AS SELECT

    CAST("Transaction ID" AS INTEGER)       AS transaction_id,
    CAST("Date" AS DATE)                    AS order_date,
    "Product Category"                      AS product_category,
    "Product Name"                          AS product_name,
    CAST("Units Sold" AS INTEGER)            AS units_sold,
    CAST("Unit Price" AS DECIMAL(12, 2))     AS unit_price,
    CAST("Total Revenue" AS DECIMAL(14, 2))  AS total_revenue,
    "Region"                                AS region,
    "Payment Method"                        AS payment_method

FROM read_csv(
    '{csv_path}',
    header = true,
    sample_size = -1
);
""")

print("View 'online_sales' created successfully.")

View 'online_sales' created successfully.


In [4]:
def sql(query: str):
    """Run a SQL query and return the result as a DataFrame."""
    return con.sql(query).df()

## 1. Data Exploration

First, inspect the dataset structure and sample records.

In [5]:
sql("""
SELECT * FROM online_sales
LIMIT 10;
""")

,transaction_id,order_date,product_category,product_name,units_sold,unit_price,total_revenue,region,payment_method
0,10001,2024-01-01,Electronics,iPhone 14 Pro,2,999.99,1999.98,North America,Credit Card
1,10002,2024-01-02,Home Appliances,Dyson V11 Vacuum,1,499.99,499.99,Europe,PayPal
2,10003,2024-01-03,Clothing,Levi's 501 Jeans,3,69.99,209.97,Asia,Debit Card
3,10004,2024-01-04,Books,The Da Vinci Code,4,15.99,63.96,North America,Credit Card
4,10005,2024-01-05,Beauty Products,Neutrogena Skincare Set,1,89.99,89.99,Europe,PayPal
5,10006,2024-01-06,Sports,Wilson Evolution Basketball,5,29.99,149.95,Asia,Credit Card
6,10007,2024-01-07,Electronics,MacBook Pro 16-inch,1,2499.99,2499.99,North America,Credit Card
7,10008,2024-01-08,Home Appliances,Blueair Classic 480i,2,599.99,1199.98,Europe,PayPal
8,10009,2024-01-09,Clothing,Nike Air Force 1,6,89.99,539.94,Asia,Debit Card
9,10010,2024-01-10,Books,Dune by Frank Herbert,2,25.99,51.98,North America,Credit Card


In [6]:
sql("""DESCRIBE online_sales;""")

,column_name,column_type,null,key,default,extra
0,transaction_id,INTEGER,YES,None,None,None
1,order_date,DATE,YES,None,None,None
2,product_category,VARCHAR,YES,None,None,None
3,product_name,VARCHAR,YES,None,None,None
4,units_sold,INTEGER,YES,None,None,None
5,unit_price,"DECIMAL(12,2)",YES,None,None,None
6,total_revenue,"DECIMAL(14,2)",YES,None,None,None
7,region,VARCHAR,YES,None,None,None
8,payment_method,VARCHAR,YES,None,None,None


In [7]:
sql("""
SUMMARIZE online_sales;
""")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,transaction_id,INTEGER,10001,10240,246,10120.5,69.42621983083913,10060,10120,10180,240,0.0
1,order_date,DATE,2024-01-01,2024-08-27,286,2024-04-29 12:00:00,None,2024-02-29,2024-04-29,2024-06-28,240,0.0
2,product_category,VARCHAR,Beauty Products,Sports,6,None,None,None,None,None,240,0.0
3,product_name,VARCHAR,1984 by George Orwell,iRobot Braava Jet M6,227,None,None,None,None,None,240,0.0
4,units_sold,INTEGER,1,10,7,2.158333333333333,1.3224538706088862,1,2,3,240,0.0
5,unit_price,"DECIMAL(12,2)",6.50,3899.99,118,236.39558333333332,429.44669491047745,30,91,258,240,0.0
6,total_revenue,"DECIMAL(14,2)",6.50,3899.99,210,335.699375,485.8044685651322,62,177,399,240,0.0
7,region,VARCHAR,Asia,North America,3,None,None,None,None,None,240,0.0
8,payment_method,VARCHAR,Credit Card,PayPal,3,None,None,None,None,None,240,0.0


The dataset contains transaction-level online sales records covering the
period shown above. Each transaction is identified by a unique transaction ID.

In [8]:
sql("""
SELECT
    transaction_id,
    COUNT(*) AS occurrence_count
FROM online_sales
GROUP BY transaction_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;
""")

,transaction_id,occurrence_count


In [9]:
sql("""
SELECT
    COUNT(*) AS revenue_mismatch_count
FROM online_sales
WHERE ABS(
    total_revenue - (units_sold * unit_price)
) > 0.01;
""")

,revenue_mismatch_count
0,0


## 2. Sales Performance Overview

This section summarizes the primary sales KPIs, including revenue,
units sold, transaction volume, and average transaction value.

In [10]:
sql("""
SELECT
    COUNT(*) AS total_transactions,
    SUM(units_sold) AS total_units_sold,
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    ROUND(AVG(total_revenue), 2) AS average_transaction_value,
    ROUND(AVG(units_sold), 2) AS average_units_per_transaction
FROM online_sales;
""")

,total_transactions,total_units_sold,total_revenue,average_transaction_value,average_units_per_transaction
0,240,518.0,80567.85,335.7,2.16


In [11]:
sql("""
SELECT
    STRFTIME(DATE_TRUNC('month', order_date), '%Y-%m') AS sales_month,
    COUNT(*) AS transaction_count,
    SUM(units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS total_revenue
FROM online_sales
GROUP BY sales_month
ORDER BY sales_month;
""")

,sales_month,transaction_count,units_sold,total_revenue
0,2024-01,31,68.0,14548.32
1,2024-02,29,77.0,10803.37
2,2024-03,31,82.0,12849.24
3,2024-04,30,65.0,12451.69
4,2024-05,31,60.0,8455.49
5,2024-06,30,61.0,7384.55
6,2024-07,31,53.0,6797.08
7,2024-08,27,52.0,7278.11


In [12]:
sql("""
WITH monthly_sales AS (
    SELECT
        STRFTIME(DATE_TRUNC('month', order_date), '%Y-%m') AS sales_month,
        SUM(total_revenue) AS total_revenue
    FROM online_sales
    GROUP BY sales_month
),

monthly_comparison AS (
    SELECT
        sales_month,
        total_revenue,
        LAG(total_revenue) OVER (
            ORDER BY sales_month
        ) AS previous_month_revenue
    FROM monthly_sales
)

SELECT
    sales_month,
    ROUND(total_revenue, 2) AS total_revenue,
    ROUND(previous_month_revenue, 2) AS previous_month_revenue,

    ROUND(
        100.0 * (
            total_revenue - previous_month_revenue
        ) / NULLIF(previous_month_revenue, 0),
        2
    ) AS month_over_month_growth_pct

FROM monthly_comparison
ORDER BY sales_month;
""")

,sales_month,total_revenue,previous_month_revenue,month_over_month_growth_pct
0,2024-01,14548.32,NaN,NaN
1,2024-02,10803.37,14548.32,-25.74
2,2024-03,12849.24,10803.37,18.94
3,2024-04,12451.69,12849.24,-3.09
4,2024-05,8455.49,12451.69,-32.09
5,2024-06,7384.55,8455.49,-12.67
6,2024-07,6797.08,7384.55,-7.96
7,2024-08,7278.11,6797.08,7.08


In [13]:
sql("""
SELECT
    product_category,
    COUNT(*) AS transaction_count,
    SUM(units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    ROUND(AVG(total_revenue), 2) AS average_transaction_value,

    ROUND(
        100.0 * SUM(total_revenue)
        / SUM(SUM(total_revenue)) OVER (),
        2
    ) AS revenue_share_pct

FROM online_sales
GROUP BY product_category
ORDER BY total_revenue DESC;
""")

,product_category,transaction_count,units_sold,total_revenue,average_transaction_value,revenue_share_pct
0,Electronics,40,66.0,34982.41,874.56,43.42
1,Home Appliances,40,59.0,18646.16,466.15,23.14
2,Sports,40,88.0,14326.52,358.16,17.78
3,Clothing,40,145.0,8128.93,203.22,10.09
4,Beauty Products,40,46.0,2621.90,65.55,3.25
5,Books,40,114.0,1861.93,46.55,2.31


In [14]:
sql("""
SELECT
    region,
    COUNT(*) AS transaction_count,
    SUM(units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    ROUND(AVG(total_revenue), 2) AS average_transaction_value
FROM online_sales
GROUP BY region
ORDER BY total_revenue DESC;
""")

,region,transaction_count,units_sold,total_revenue,average_transaction_value
0,North America,80,180.0,36844.34,460.55
1,Asia,80,233.0,22455.45,280.69
2,Europe,80,105.0,21268.06,265.85


In [15]:
sql("""
SELECT
    region,
    product_category,
    SUM(units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS total_revenue
FROM online_sales
GROUP BY
    region,
    product_category
ORDER BY
    region,
    total_revenue DESC;
""")

,region,product_category,units_sold,total_revenue
0,Asia,Sports,88.0,14326.52
1,Asia,Clothing,145.0,8128.93
2,Europe,Home Appliances,59.0,18646.16
3,Europe,Beauty Products,46.0,2621.90
4,North America,Electronics,66.0,34982.41
5,North America,Books,114.0,1861.93


In [16]:
sql("""
SELECT
    payment_method,
    COUNT(*) AS transaction_count,
    SUM(units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    ROUND(AVG(total_revenue), 2) AS average_transaction_value
FROM online_sales
GROUP BY payment_method
ORDER BY total_revenue DESC;
""")

,payment_method,transaction_count,units_sold,total_revenue,average_transaction_value
0,Credit Card,120,268.0,51170.86,426.42
1,PayPal,80,105.0,21268.06,265.85
2,Debit Card,40,145.0,8128.93,203.22


In [17]:
sql("""
SELECT
    product_name,
    product_category,
    SUM(units_sold) AS units_sold,
    ROUND(SUM(total_revenue), 2) AS total_revenue
FROM online_sales
GROUP BY
    product_name,
    product_category
ORDER BY total_revenue DESC
LIMIT 10;
""")

,product_name,product_category,units_sold,total_revenue
0,Canon EOS R5 Camera,Electronics,1.0,3899.99
1,LG OLED TV,Home Appliances,2.0,2599.98
2,MacBook Pro 16-inch,Electronics,1.0,2499.99
3,Apple MacBook Pro 16-inch,Electronics,1.0,2399.00
4,iPhone 14 Pro,Electronics,2.0,1999.98
5,Peloton Bike,Sports,1.0,1895.00
6,HP Spectre x360 Laptop,Electronics,1.0,1599.99
7,Roomba i7+,Home Appliances,2.0,1599.98
8,Samsung Odyssey G9 Gaming Monitor,Electronics,1.0,1499.99
9,Samsung Galaxy Tab S8,Electronics,2.0,1499.98


In [18]:
sql("""
WITH product_performance AS (
    SELECT
        product_category,
        product_name,
        SUM(units_sold) AS units_sold,
        SUM(total_revenue) AS total_revenue
    FROM online_sales
    GROUP BY
        product_category,
        product_name
),

ranked_products AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY product_category
            ORDER BY total_revenue DESC
        ) AS revenue_rank
    FROM product_performance
)

SELECT
    product_category,
    product_name,
    units_sold,
    ROUND(total_revenue, 2) AS total_revenue
FROM ranked_products
WHERE revenue_rank = 1
ORDER BY total_revenue DESC;
""")

,product_category,product_name,units_sold,total_revenue
0,Electronics,Canon EOS R5 Camera,1.0,3899.99
1,Home Appliances,LG OLED TV,2.0,2599.98
2,Sports,Peloton Bike,1.0,1895.00
3,Clothing,Nike Air Force 1,6.0,539.94
4,Beauty Products,Dyson Supersonic Hair Dryer,1.0,399.99
5,Books,The Silent Patient by Alex Michaelides,5.0,134.95


## Key Findings

1. **Revenue Performance**  
   Total revenue reached `$80567.85` from `240` transactions during the
   observed period.

2. **Leading Category**  
   `Electronics` generated the highest revenue and accounted for approximately
   `43.42%` of total sales.

3. **Regional Performance**  
   `North America` was the highest-revenue region, while `Asia` recorded the
   highest average transaction value.

4. **Payment Behavior**  
   `Credit Card` was the most frequently recorded payment method in the dataset.

5. **Top Product**  
   `Canon EOS R5 Camera` was the highest-revenue product during the observed period.

## Limitations

- The dataset represents a limited observation period.
- It does not contain customer identifiers, costs, discounts, or profit data.
- Revenue should not be interpreted as profit.
- Customer retention and repeat-purchase behavior cannot be measured.
- The analysis is intended as a portfolio demonstration rather than a
  real-world business recommendation.